# Chaînes de Markov

| Module | Durée | Prérequis |
|---|---|---|
| M7 – Valeurs propres et dynamique | 75 min | M5 |

In [ ]:
# Style graphique Liouaï : couleurs du logo, axes sobres
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

LIOUAI = {"navy": "#243743", "vert": "#28B78D", "vert_texte": "#167A5C", "gris": "#A1A9AE",
          "gris_texte": "#5C6870", "jaune": "#E5C800", "fond": "#F3F5F6"}
plt.rcParams.update({
    "axes.prop_cycle": plt.cycler(color=[LIOUAI["vert"], LIOUAI["navy"], LIOUAI["gris"],
                                         LIOUAI["jaune"], LIOUAI["vert_texte"], LIOUAI["gris_texte"]]),
    "axes.edgecolor": LIOUAI["gris"], "axes.labelcolor": LIOUAI["navy"],
    "axes.titlecolor": LIOUAI["navy"], "axes.titleweight": "bold",
    "axes.spines.top": False, "axes.spines.right": False,
    "xtick.color": LIOUAI["gris_texte"], "ytick.color": LIOUAI["gris_texte"],
    "text.color": LIOUAI["navy"], "grid.color": "#E1E6E9", "legend.frameon": False,
    "font.family": "sans-serif", "font.sans-serif": ["Roboto", "Helvetica", "Arial", "DejaVu Sans"],
    "figure.dpi": 100,
})
# Dégradé de marque : gris clair -> vert -> navy (et sa version inversée)
if "liouai" not in mpl.colormaps:
    _cmap = LinearSegmentedColormap.from_list("liouai", [LIOUAI["fond"], LIOUAI["vert"], LIOUAI["navy"]])
    mpl.colormaps.register(_cmap)
    mpl.colormaps.register(_cmap.reversed())

## ① Définir la question

**Première situation.** Un camping de la côte bretonne veut indiquer sur son site la proportion de journées ensoleillées, nuageuses et pluvieuses qu'un vacancier peut attendre. Un passionné de météo a noté le temps de chaque jour pendant plusieurs étés. Il en a tiré des **probabilités de transition** : par exemple, après une journée de soleil, le lendemain est ensoleillé 7 fois sur 10.

**Seconde situation.** Trois opérateurs de téléphonie se partagent le marché d'une ville. Chaque année, une partie des clients change d'opérateur ; une étude de marché a estimé, pour chaque opérateur, la proportion de clients qui restent et celle qui partent chez chacun des concurrents.

**Les questions : à long terme, quelle proportion de jours de chaque type ? quelles parts de marché ? Ces proportions dépendent-elles de la situation de départ ?**

**Activité** : Les deux situations ont la même structure mathématique. Laquelle ? Qu'est-ce qui joue le rôle du « temps qu'il fait » dans la seconde ?

Indiquez vos réponses et justifiez-les.

## ② Estimer avant de calculer

In [ ]:
pari_proportion_soleil = 0.5       # à long terme (modifiez)
pari_depend_du_depart = "oui"      # "oui" ou "non" (modifiez)

## ③ Abstraire : une chaîne de Markov

On décrit l'état du système par un **vecteur de probabilités** $\mathbf{p}_t$ (de somme 1). Pour la météo, $\mathbf{p}_t = (p_{\text{soleil}}\,;\,p_{\text{nuages}}\,;\,p_{\text{pluie}})$. Le passage d'un jour au suivant ne dépend que du jour présent : c'est une **chaîne de Markov**.

![Schéma : trois états météo reliés par des flèches portant les probabilités de transition ; la matrice P a pour colonne j les probabilités de partir de l'état j](../../assets/schemas/schema-markov.svg)

$$
\mathbf{p}_{t+1} = P\,\mathbf{p}_t .
$$

Si les proportions se stabilisent, c'est sur un vecteur $\boldsymbol{\pi}$ qui ne change plus :

$$
P\,\boldsymbol{\pi} = \boldsymbol{\pi} .
$$

C'est un **vecteur propre de $P$ pour la valeur propre 1**, appelé **état stationnaire**. Comme chaque colonne de $P$ a pour somme 1, la valeur propre 1 existe toujours, et aucune autre valeur propre n'a un module plus grand.

**Le concept s'applique-t-il toujours ?** Il faut que le futur ne dépende que du présent (et pas des jours précédents), et que les probabilités de transition ne changent pas avec le temps. Nous verrons aussi que l'état stationnaire n'est pas toujours unique, ni toujours atteint.

**Activité** : Pour une chaîne à deux états, de matrice $\begin{pmatrix} 0{,}9 & 0{,}5 \\ 0{,}1 & 0{,}5 \end{pmatrix}$, résolvez $P\boldsymbol{\pi} = \boldsymbol{\pi}$ avec $\pi_1 + \pi_2 = 1$. Vous devez trouver $\boldsymbol{\pi} = (5/6\,;\,1/6)$.

Indiquez vos réponses et justifiez-les.

## ④ Calculer

### La météo, jour après jour

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

etats = ["soleil", "nuages", "pluie"]
P = np.array([[0.7, 0.3, 0.2],
              [0.2, 0.4, 0.4],
              [0.1, 0.3, 0.4]])
print("Sommes des colonnes :", P.sum(axis=0))

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5), sharey=True)
for ax, depart in zip(axes, range(3)):
    p = np.eye(3)[depart]
    historique = [p]
    for _ in range(12):
        p = P @ p
        historique.append(p)
    historique = np.array(historique)
    for k, couleur in zip(range(3), [LIOUAI["jaune"], LIOUAI["gris"], LIOUAI["vert"]]):
        ax.plot(historique[:, k], "o-", ms=3, color=couleur, label=etats[k])
    ax.set_title(f"aujourd'hui : {etats[depart]}"); ax.set_xlabel("jours"); ax.grid(alpha=0.3)
axes[0].set_ylabel("probabilité"); axes[0].legend()
plt.tight_layout(); plt.show()
print("Après 12 jours, en partant du soleil :", historique[-1].round(4))

### L'état stationnaire, vecteur propre

In [ ]:
def en_clair(noms, valeurs):
    return ", ".join(f"{n} {v:.1%}" for n, v in zip(noms, valeurs))

def etat_stationnaire_eig(P):
    valeurs, vecteurs = np.linalg.eig(P)
    k = np.argmin(np.abs(valeurs - 1))
    v = vecteurs[:, k].real
    return v / v.sum()

valeurs = np.linalg.eigvals(P)
pi = etat_stationnaire_eig(P)
print("Valeurs propres de P :", np.round(np.real_if_close(valeurs), 4))
print("État stationnaire    :", en_clair(etats, pi))
print("P pi = pi ?          ", np.allclose(P @ pi, pi))

### Corroborer par une simulation

On simule 20 000 jours de météo au hasard, en suivant les probabilités de transition, et l'on compte les fréquences.

In [ ]:
rng = np.random.default_rng(7)
jour = 0
compte = np.zeros(3)
for _ in range(20_000):
    jour = rng.choice(3, p=P[:, jour])       # colonne du jour présent : probabilités du lendemain
    compte[jour] += 1
print("Fréquences simulées :", en_clair(etats, compte / compte.sum()))

### À quelle vitesse oublie-t-on le départ ?

L'écart à l'état stationnaire décroît comme $\lvert \lambda_2 \rvert^t$, où $\lambda_2$ est la deuxième plus grande valeur propre en module.

In [ ]:
lam2 = sorted(np.abs(valeurs))[-2]
p = np.array([0., 0, 1])
ecarts = []
for t in range(15):
    ecarts.append(np.abs(p - pi).sum())
    p = P @ p
fig, ax = plt.subplots(figsize=(7, 3.4))
ax.semilogy(ecarts, "o", color=LIOUAI["navy"], label="écart mesuré")
ax.semilogy(ecarts[0] * lam2 ** np.arange(15), "--", color=LIOUAI["vert"], label=f"pente |λ2|^t, |λ2| = {lam2:.3f}")
ax.set_xlabel("jours"); ax.set_ylabel("écart à l'état stationnaire"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

### Les parts de marché

In [ ]:
operateurs = ["Azur", "Boréal", "Cyan"]
Q = np.array([[0.85, 0.10, 0.05],       # colonne j : devenir des clients de l'opérateur j
              [0.10, 0.80, 0.15],
              [0.05, 0.10, 0.80]])
parts_actuelles = np.array([0.50, 0.30, 0.20])
print("Parts actuelles     :", en_clair(operateurs, parts_actuelles))
print("Parts à long terme  :", en_clair(operateurs, etat_stationnaire_eig(Q)))

Q_promo = Q.copy()
Q_promo[:, 2] = [0.03, 0.07, 0.90]       # Cyan fidélise mieux ses clients
print("Si Cyan fidélise    :", en_clair(operateurs, etat_stationnaire_eig(Q_promo)))

### Quand le concept ne s'applique pas comme prévu

In [ ]:
contre_exemples = {
    "alternance stricte": np.array([[0., 1], [1, 0]]),
    "état absorbant": np.array([[1., 0.3], [0, 0.7]]),
    "deux mondes séparés": np.array([[0.6, 0.4, 0, 0], [0.4, 0.6, 0, 0], [0, 0, 0.5, 0.5], [0, 0, 0.5, 0.5]]),
}
for nom, M in contre_exemples.items():
    print(f"{nom:20s} valeurs propres : {np.round(np.real_if_close(np.linalg.eigvals(M)), 3)}")

p = np.array([1., 0])
print("\nAlternance, en partant de l'état 1 :",
      [(np.linalg.matrix_power(contre_exemples["alternance stricte"], t) @ p).astype(int).tolist() for t in range(5)])

**Activité** : Pour chaque contre-exemple, que se passe-t-il à long terme ? L'état stationnaire existe-t-il ? Est-il unique ? Est-il atteint quel que soit le départ ? Quelle valeur propre « coupable » repérez-vous ?

Indiquez vos réponses et justifiez-les.

Un résultat (le théorème de **Perron-Frobenius**, admis ici) garantit que tout se passe bien dans un cas simple : si **tous** les coefficients de $P$ sont strictement positifs, l'état stationnaire est unique et il est atteint quel que soit le départ.

## ⑤ Interpréter

In [ ]:
print(f"Votre pari : {pari_proportion_soleil:.0%} de soleil ; dépend du départ : {pari_depend_du_depart}")
print(f"Résultat   : {pi[0]:.0%} de soleil, quel que soit le temps du premier jour")

**Activité** :

1. Rédigez la phrase que le camping peut afficher sur son site, honnêtement : que garantit-elle, que ne garantit-elle pas ?
2. Combien de jours faut-il pour que la météo d'aujourd'hui n'ait presque plus d'influence ? Reliez la réponse à $\lambda_2$.
3. Dites à l'opérateur Cyan ce que lui rapporterait, à long terme, une meilleure fidélisation. Pourquoi la répartition actuelle des clients n'a-t-elle aucune importance pour la réponse ?
4. Quelle conclusion générale tirez-vous pour **tous** les systèmes de ce type ?

Indiquez vos réponses et justifiez-les.

## ⑥ Critiquer et généraliser

- **L'hypothèse de Markov.** La météo d'après-demain dépend-elle vraiment seulement de celle de demain ? Comment tester cette hypothèse sur les relevés ?
- **Les probabilités sont estimées.** Elles proviennent de quelques étés de relevés. Si chacune est connue à $\pm 0{,}05$ près, de combien la proportion de soleil peut-elle varier ?
- **Généraliser.** Écrivez une fonction `etat_stationnaire(P)` qui vérifie que $P$ est bien une matrice de transition (coefficients positifs, colonnes de somme 1), puis calcule l'état stationnaire **sans** `eig` : en résolvant le système $(P - I)\boldsymbol{\pi} = \mathbf{0}$ complété par l'équation $\pi_1 + \dots + \pi_n = 1$ (module M6). Comparez avec `etat_stationnaire_eig`.

In [ ]:
def etat_stationnaire(P):
    # À compléter
    ...

## Corrigé

*À n'ouvrir qu'après avoir cherché.*

In [ ]:
def etat_stationnaire(P):
    P = np.asarray(P, dtype=float)
    if (P < 0).any() or not np.allclose(P.sum(axis=0), 1):
        raise ValueError("Ce n'est pas une matrice de transition : coefficients négatifs ou colonnes de somme différente de 1.")
    n = len(P)
    A = np.vstack([P - np.eye(n), np.ones(n)])           # n équations (P - I) pi = 0, plus la somme = 1
    b = np.r_[np.zeros(n), 1]
    pi, *_ = np.linalg.lstsq(A, b, rcond=None)
    return pi

for nom, M in {"météo": P, "opérateurs": Q}.items():
    print(f"{nom:12s} sans eig : {etat_stationnaire(M).round(4)}   avec eig : {etat_stationnaire_eig(M).round(4)}")

rng = np.random.default_rng(1)
soleil = []
for _ in range(2000):
    M = np.clip(P + rng.uniform(-0.05, 0.05, P.shape), 0.01, None)
    M /= M.sum(axis=0)                                    # on renormalise chaque colonne
    soleil.append(etat_stationnaire(M)[0])
print(f"\nAvec des probabilités à ±0,05 près, la proportion de soleil varie de {min(soleil):.0%} à {max(soleil):.0%}")

## Bilan — auto-évaluation

Pour chaque compétence travaillée dans ce notebook, choisissez un niveau : **0** non observé · **1** guidé · **2** autonome · **3** transfert.
La cellule enregistre votre bilan dans `progression.json`, à la racine du site.

In [ ]:
import json, datetime, pathlib

bilan = {
    "Identifier les concepts pertinents et leurs relations": 1,
    "Tirer des conclusions générales sur un type de problème": 1,
    "Reconnaître si le concept s'applique": 1,
    "Estimer la solution avant de calculer": 1,
    "Énoncer le résultat dans le contexte du problème": 1,
}   # modifiez les niveaux

ici = pathlib.Path.cwd()
racine = next((p for p in [ici, *ici.parents] if (p / "myst.yml").exists()), ici)
fichier = racine / "progression.json"
historique = json.loads(fichier.read_text(encoding="utf-8")) if fichier.exists() else []
historique.append({"notebook": "7.3-chaines-de-markov", "date": datetime.date.today().isoformat(), "niveaux": bilan})
fichier.write_text(json.dumps(historique, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Bilan enregistré ({len(historique)} entrée(s) dans {fichier.name}).")